# VietNamNet News Classification with PhoBERT Base v2

This notebook is adapted for the `phobert-news-classification-app` monorepo. It reads data from the configured `DATASET_DIR` and writes training outputs to `train/runs/` and the deployable package to `train/artifacts/active/`.

**Task**: classify Vietnamese news articles into **19 topics**.

| Step | What it does |
|------|--------------|
| **0. Setup** | Install packages, select CUDA/MPS/CPU runtime, locate the repo root, and materialize parquet files from `dataset/data_URLs.json` if needed |
| **1. Load data** | Read 19 parquet files in small parquet batches |
| **2. EDA** | Inspect class balance, text length, and summary statistics |
| **3. Preprocess** | Use ViTokenizer; keep stopwords because BERT learns context |
| **4. Tokenize** | Use a Head-Tail strategy: first 127 tokens + last 127 tokens |
| **5. Train** | Fine-tune PhoBERT Base v2 with a weighted Trainer |
| **6. Evaluate** | Generate a classification report, confusion matrix, per-class F1, and training curves |
| **7. Export** | Save model config and metadata |
| **8. Calibrate** | Apply temperature scaling and per-class threshold tuning |
| **9. Diagnose** | Print follow-up signals for model improvement |

> **Model**: `vinai/phobert-base-v2`, `MAX_LENGTH=256`, Head-Tail truncation, balanced class weights.
>
> **Dataset**: by default the notebook downloads `dathuynh1108/vietnamnet-news` from Hugging Face into `./dataset` under the current working directory. If you provide only `data_URLs.json`, it can still crawl title/content and create parquet files.
>
> **Caching**: preprocessed text is cached in `./train/runs/temp/`; model artifacts are saved in `./train/runs/model/` and synced to `./train/artifacts/active/`.

## Colab quick start

```python
# Run this notebook from the folder you want to use as the workspace root.
# Dataset is auto-downloaded from dathuynh1108/vietnamnet-news when ./dataset is empty.
# Outputs are written under ./train/runs and ./train/artifacts.
```


---
## Section 0 - Setup
Run this section every time you open the notebook.

In [ ]:
# -- 0.1 Install packages -------------------------------------------------
# Keep this cell at the top so fresh Colab kernels can prepare dependencies.
# Colab already ships CUDA Torch/NumPy/Pandas; do not reinstall those packages here.
# This cell installs small missing dependencies and verifies they import before continuing.
import importlib.util
import os
import sys
import subprocess

INSTALL_PACKAGES = os.getenv("VNN_INSTALL_PACKAGES", "1") == "1"
PIP_REQUIREMENTS = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "pyvi": "pyvi",
    "bs4": "beautifulsoup4",
    "lxml": "lxml",
    "huggingface_hub": "huggingface_hub",
}

# These are expected to exist on Colab. We validate them in the next cell, but avoid
# reinstalling them because replacing runtime packages is a common cause of kernel restarts.
RUNTIME_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "pyarrow": "pyarrow",
    "scipy": "scipy",
    "joblib": "joblib",
    "requests": "requests",
}

missing_runtime = [pkg for mod, pkg in RUNTIME_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing_runtime:
    print("Missing base runtime packages:", ", ".join(missing_runtime))
    print("On Colab, switch to a standard Python/GPU runtime instead of reinstalling these here.")
    raise SystemExit("Base runtime packages are missing.")

missing_packages = [
    package for module, package in PIP_REQUIREMENTS.items()
    if importlib.util.find_spec(module) is None
]

if INSTALL_PACKAGES and missing_packages:
    print("Installing missing notebook dependencies:")
    print("  " + " ".join(missing_packages))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        *missing_packages,
    ])
    importlib.invalidate_caches()
    still_missing = [
        package for module, package in PIP_REQUIREMENTS.items()
        if package in missing_packages and importlib.util.find_spec(module) is None
    ]
    if still_missing:
        print("Installed packages are not importable yet:", ", ".join(still_missing))
        print("Restart the Colab runtime once, then run from cell 0.2 onward.")
        raise SystemExit("Dependencies were installed but are not importable yet.")
    print("Package installation complete and imports are ready.")
elif INSTALL_PACKAGES:
    print("All notebook dependencies are already installed.")
else:
    print("Package installation skipped because VNN_INSTALL_PACKAGES=0.")


In [ ]:
# -- 0.2 Validate imports and runtime ------------------------------------
import importlib.util
import inspect
import sys

_REQUIRED = {
    "pandas":       "pandas",
    "numpy":        "numpy",
    "torch":        "torch",
    "matplotlib":   "matplotlib",
    "seaborn":      "seaborn",
    "tqdm":         "tqdm",
    "sklearn":      "scikit-learn",
    "transformers": "transformers",
    "pyarrow":      "pyarrow",
    "scipy":        "scipy",
    "accelerate":   "accelerate",
    "pyvi":         "pyvi",
    "joblib":       "joblib",
    "requests":     "requests",
    "bs4":          "beautifulsoup4",
    "lxml":         "lxml",
    "huggingface_hub": "huggingface_hub",
}

_missing = {pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None}
if _missing:
    print("=" * 72)
    print("  CANNOT CONTINUE - Missing Python packages")
    print("=" * 72)
    print("  Missing packages:")
    for _p in sorted(_missing):
        print(f"    - {_p}")
    print()
    print("  Re-run the install cell above. If you manage torch manually, make sure")
    print("  the installed torch build supports your target backend: CUDA, MPS, or CPU.")
    print("  PyTorch install guide: https://pytorch.org/get-started/locally/")
    print("=" * 72)
    raise SystemExit("Missing dependencies. See the message above.")

# -- Imports --------------------------------------------------------------
import os, re, pickle, time, datetime, json, warnings, shutil
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm import tqdm
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, EarlyStoppingCallback, TrainerCallback, Trainer,
)
from torch.utils.data import Dataset as TorchDataset

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

# -- Runtime device -------------------------------------------------------
def _mps_available():
    return (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    )

if torch.cuda.is_available():
    DEVICE_TYPE = "cuda"
    device = "cuda"
    CUDA_DEVICE_COUNT = torch.cuda.device_count()
    CUDA_DEVICE_NAMES = [torch.cuda.get_device_name(i) for i in range(CUDA_DEVICE_COUNT)]
    device_name = CUDA_DEVICE_NAMES[0]
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    cc_major = torch.cuda.get_device_properties(0).major
    cc_minor = torch.cuda.get_device_properties(0).minor
    compute_cap = cc_major + cc_minor / 10
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif _mps_available():
    DEVICE_TYPE = "mps"
    device = "mps"
    CUDA_DEVICE_COUNT = 0
    CUDA_DEVICE_NAMES = []
    device_name = "Apple Metal Performance Shaders"
    vram_gb = None
    cc_major = cc_minor = 0
    compute_cap = 0
else:
    DEVICE_TYPE = "cpu"
    device = "cpu"
    CUDA_DEVICE_COUNT = 0
    CUDA_DEVICE_NAMES = []
    device_name = "CPU"
    vram_gb = None
    cc_major = cc_minor = 0
    compute_cap = 0

print(f"Dependencies ready - Python {sys.version.split()[0]}")
print(f"Runtime device: {DEVICE_TYPE.upper()} - {device_name}")
if DEVICE_TYPE == "cuda":
    print(f"GPU     : {device_name}  ({vram_gb:.1f} GB VRAM)  Compute {cc_major}.{cc_minor}  CUDA {torch.version.cuda}")
    print(f"CUDA GPUs visible: {CUDA_DEVICE_COUNT} - {', '.join(CUDA_DEVICE_NAMES)}")
elif DEVICE_TYPE == "mps":
    print("MPS note: training can run, but CUDA-only optimizations are disabled.")
else:
    print("CPU note: this keeps the notebook runnable, but full PhoBERT training will be very slow.")
print(f"PyTorch : {torch.__version__}")



In [ ]:
# -- 0.3 Paths and training configuration --------------------------------
# Workspace setup:
# - Run this notebook from the folder you want as the workspace root.
# - The notebook creates ./dataset, ./train/runs, and ./train/artifacts automatically.
# - If you only provide data_URLs.json and need crawling, place materialize_vietnamnet_dataset.py
#   in ./materialize_vietnamnet_dataset.py or ./train/scripts/materialize_vietnamnet_dataset.py.

# Defaults are relative to the current working directory. Override only when needed.
REPO_DIR = os.getenv("VNN_REPO_ROOT", str(Path.cwd()))
DATASET_DIR = os.getenv("VNN_DATASET_DIR", str(Path(REPO_DIR).expanduser().resolve() / "dataset"))
HF_DATASET_REPO = os.getenv("VNN_HF_DATASET_REPO", "dathuynh1108/vietnamnet-news")
AUTO_DOWNLOAD_DATASET = os.getenv("VNN_AUTO_DOWNLOAD_DATASET", "1") == "1"

REPO_ROOT = Path(REPO_DIR).expanduser().resolve()
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd().resolve()

DATASET_FOLDER = Path(DATASET_DIR).expanduser().resolve()
if not DATASET_FOLDER.exists():
    DATASET_FOLDER = REPO_ROOT / "dataset"

NOTEBOOK_DIR   = REPO_ROOT / "train" / "notebooks"
TEMP_DIR       = REPO_ROOT / "train" / "runs" / "temp"
RESULTS_DIR    = REPO_ROOT / "train" / "runs" / "results"
MODEL_DIR      = REPO_ROOT / "train" / "runs" / "model"
ARTIFACTS_DIR  = REPO_ROOT / "train" / "artifacts"
ACTIVE_ARTIFACT_DIR = ARTIFACTS_DIR / "active"
SCRIPT_DIR     = REPO_ROOT / "train" / "scripts"
if not (SCRIPT_DIR / "materialize_vietnamnet_dataset.py").exists() and (REPO_ROOT / "materialize_vietnamnet_dataset.py").exists():
    SCRIPT_DIR = REPO_ROOT
DATA_URLS_PATH = DATASET_FOLDER / "data_URLs.json"

for _d in [DATASET_FOLDER, TEMP_DIR, RESULTS_DIR, MODEL_DIR, ACTIVE_ARTIFACT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_PATH = str(TEMP_DIR / "processed_data.pkl")
PROCESSED_META_PATH = str(TEMP_DIR / "processed_data.meta.json")
HEADTAIL_TRAIN_CACHE_PATH = str(TEMP_DIR / "headtail_train.pkl")
HEADTAIL_TEST_CACHE_PATH = str(TEMP_DIR / "headtail_test.pkl")
CHECKPOINT_DIR = str(TEMP_DIR / "checkpoints")
TRAIN_HISTORY_PATH  = str(MODEL_DIR / "train_history.pkl")
MODEL_CONFIG_PATH = str(MODEL_DIR / "config.json")
LABEL_CONFIG_PATH   = str(MODEL_DIR / "label_config.json")
THRESHOLD_FILE_NAME = "thresholds.json"
THRESHOLD_PATH = str(MODEL_DIR / THRESHOLD_FILE_NAME)

# -- Dataset materialization ---------------------------------------------
AUTO_MATERIALIZE_DATASET = True
CRAWL_WORKERS            = int(os.getenv("VNN_CRAWL_WORKERS", "24"))
CRAWL_BATCH_SIZE         = int(os.getenv("VNN_CRAWL_BATCH_SIZE", "128"))
_MAX_URLS_ENV            = os.getenv("VNN_MAX_URLS_PER_CATEGORY", "").strip()
URLS_PER_CATEGORY        = int(_MAX_URLS_ENV) if _MAX_URLS_ENV else None

# -- Colab-safe data loading ---------------------------------------------
PARQUET_BATCH_SIZE = int(os.getenv("VNN_PARQUET_BATCH_SIZE", "20480"))
_MAX_ROWS_CLASS_ENV = os.getenv("VNN_MAX_ROWS_PER_CLASS", "0").strip()
MAX_ROWS_PER_CLASS = int(_MAX_ROWS_CLASS_ENV) if _MAX_ROWS_CLASS_ENV else None
if MAX_ROWS_PER_CLASS <= 0:
    MAX_ROWS_PER_CLASS = None
RAW_CONTENT_HEAD_WORDS = int(os.getenv("VNN_RAW_CONTENT_HEAD_WORDS", "220"))
RAW_CONTENT_TAIL_WORDS = int(os.getenv("VNN_RAW_CONTENT_TAIL_WORDS", "80"))
PREPROCESS_BATCH_SIZE = int(os.getenv("VNN_PREPROCESS_BATCH_SIZE", "512"))
PREPROCESS_N_JOBS = int(os.getenv("VNN_PREPROCESS_N_JOBS", "2"))
TOKEN_LENGTH_SAMPLE_SIZE = int(os.getenv("VNN_TOKEN_LENGTH_SAMPLE_SIZE", "1000"))
SMOKE_SAMPLE_SIZE = int(os.getenv("VNN_SMOKE_SAMPLE_SIZE", "3"))
PROCESSED_CACHE_META = {
    "version": 2,
    "max_rows_per_class": MAX_ROWS_PER_CLASS,
    "raw_content_head_words": RAW_CONTENT_HEAD_WORDS,
    "raw_content_tail_words": RAW_CONTENT_TAIL_WORDS,
}
_rows_cap_label = f"{MAX_ROWS_PER_CLASS:,}" if MAX_ROWS_PER_CLASS else "full dataset"

print(f"Repo root     : {REPO_ROOT}")
print(f"Dataset folder: {DATASET_FOLDER}")
print(f"HF dataset    : {HF_DATASET_REPO if AUTO_DOWNLOAD_DATASET else 'disabled'}")
print(f"URL dataset   : {DATA_URLS_PATH if DATA_URLS_PATH.exists() else 'missing'}")
print(f"Script dir     : {SCRIPT_DIR if (SCRIPT_DIR / 'materialize_vietnamnet_dataset.py').exists() else 'missing materialize script'}")
print(f"Active artifact: {ACTIVE_ARTIFACT_DIR}")
print(f"Rows/class cap : {_rows_cap_label}")
print(f"Parquet batch  : {PARQUET_BATCH_SIZE:,} rows")
print(f"Text head/tail : {RAW_CONTENT_HEAD_WORDS}+{RAW_CONTENT_TAIL_WORDS} words")
print(f"Preprocess     : {PREPROCESS_BATCH_SIZE:,} rows/batch | {PREPROCESS_N_JOBS} jobs")
print(f"Sample checks   : token_len={TOKEN_LENGTH_SAMPLE_SIZE:,} | smoke={SMOKE_SAMPLE_SIZE}")

# -- Training knobs -------------------------------------------------------
MODEL_NAME   = 'vinai/phobert-base-v2'
MAX_LENGTH   = 256
BATCH_SIZE   = 32
GRAD_ACCUM   = 2
EVAL_BATCH   = 64
LR           = 2e-5
NUM_EPOCHS   = 7
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200
TEST_SIZE    = 0.15
RANDOM_STATE = 42

_bf16_ok = (DEVICE_TYPE == "cuda" and compute_cap >= 8.0)   # Ampere or newer
_fp16_ok = (DEVICE_TYPE == "cuda" and compute_cap >= 7.0)   # Turing or newer
BF16 = _bf16_ok
FP16 = not BF16 and _fp16_ok
PIN_MEMORY = DEVICE_TYPE == "cuda"
OPTIMIZER_NAME = "adamw_torch_fused" if DEVICE_TYPE == "cuda" else "adamw_torch"

# Conservative Colab/runtime presets. Env vars below can override these values.
TRAIN_DEVICE_COUNT = CUDA_DEVICE_COUNT if DEVICE_TYPE == "cuda" else 1
if DEVICE_TYPE == "cuda":
    if vram_gb >= 20:
        BATCH_SIZE, _single_gpu_accum, EVAL_BATCH = 32, 2, 64
    elif vram_gb >= 12:
        BATCH_SIZE, _single_gpu_accum, EVAL_BATCH = 16, 4, 32
    elif vram_gb >= 8:
        BATCH_SIZE, _single_gpu_accum, EVAL_BATCH = 8, 8, 16
    else:
        BATCH_SIZE, _single_gpu_accum, EVAL_BATCH = 4, 16, 8
    GRAD_ACCUM = max(1, _single_gpu_accum // max(TRAIN_DEVICE_COUNT, 1))
elif DEVICE_TYPE == "mps":
    BATCH_SIZE = min(BATCH_SIZE, 8)
    GRAD_ACCUM = max(GRAD_ACCUM, 4)
    EVAL_BATCH = min(EVAL_BATCH, 16)
elif DEVICE_TYPE == "cpu":
    BATCH_SIZE = min(BATCH_SIZE, 4)
    GRAD_ACCUM = max(GRAD_ACCUM, 8)
    EVAL_BATCH = min(EVAL_BATCH, 8)

BATCH_SIZE = int(os.getenv("VNN_BATCH_SIZE", str(BATCH_SIZE)))
GRAD_ACCUM = int(os.getenv("VNN_GRAD_ACCUM", str(GRAD_ACCUM)))
EVAL_BATCH = int(os.getenv("VNN_EVAL_BATCH", str(EVAL_BATCH)))
NUM_EPOCHS = int(os.getenv("VNN_NUM_EPOCHS", str(NUM_EPOCHS)))
GLOBAL_EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM * TRAIN_DEVICE_COUNT
LOGGING_STEPS = int(os.getenv("VNN_LOGGING_STEPS", "100"))
SAVE_TOTAL_LIMIT = int(os.getenv("VNN_SAVE_TOTAL_LIMIT", "1"))
DATALOADER_NUM_WORKERS = int(os.getenv("VNN_DATALOADER_NUM_WORKERS", "0"))
EARLY_STOPPING_PATIENCE = int(os.getenv("VNN_EARLY_STOPPING_PATIENCE", "3"))
GRADIENT_CHECKPOINTING = os.getenv("VNN_GRADIENT_CHECKPOINTING", "0") == "1"
TORCH_COMPILE = os.getenv("VNN_TORCH_COMPILE", "0") == "1"

TEMPERATURE_SEARCH_BOUNDS = (
    float(os.getenv("VNN_TEMPERATURE_MIN", "0.5")),
    float(os.getenv("VNN_TEMPERATURE_MAX", "5.0")),
)
ECE_N_BINS = int(os.getenv("VNN_ECE_N_BINS", "15"))
THRESHOLD_SEARCH_MIN = float(os.getenv("VNN_THRESHOLD_SEARCH_MIN", "0.3"))
THRESHOLD_SEARCH_MAX = float(os.getenv("VNN_THRESHOLD_SEARCH_MAX", "3.5"))
THRESHOLD_SEARCH_STEPS = int(os.getenv("VNN_THRESHOLD_SEARCH_STEPS", "200"))
THRESHOLD_SEARCH_PASSES = int(os.getenv("VNN_THRESHOLD_SEARCH_PASSES", "3"))
THRESHOLD_IMPROVEMENT_EPS = float(os.getenv("VNN_THRESHOLD_IMPROVEMENT_EPS", "1e-5"))
_TRAINING_ARGS_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
EVAL_STRATEGY_KWARG = {"eval_strategy": "epoch"} if "eval_strategy" in _TRAINING_ARGS_PARAMS else {"evaluation_strategy": "epoch"}

# Small recall-oriented boosts applied after balanced class weights.
MANUAL_CLASS_BOOST = {
    'Dân tộc - Tôn giáo':     1.35,
    'Kinh doanh':             1.15,
    'Thời sự':                1.15,
    'Thị trường tiêu dùng':   1.10,
    'Đời sống':               1.10,
}

print()
print(f'Runtime device: {DEVICE_TYPE.upper()} - {device_name}')
if DEVICE_TYPE == "cuda":
    _tiers = [
        (24,  'vinai/phobert-large',   64,  1, 128, True,      False,    '>= 24 GB (RTX 3090/4090)'),
        (16,  'vinai/phobert-large',   32,  2,  64, _bf16_ok,  not _bf16_ok and _fp16_ok, '16-23 GB (RTX 3080Ti/4080)'),
        (10,  'vinai/phobert-base-v2', 32,  2,  64, _bf16_ok,  not _bf16_ok and _fp16_ok, '10-15 GB (RTX 3080 10GB / A3000 12GB)'),
        ( 6,  'vinai/phobert-base-v2', 16,  4,  32, False,     _fp16_ok, '6-9 GB (RTX 3060 / 2060)'),
        ( 0,  'vinai/phobert-base-v2',  8,  8,  16, False,     _fp16_ok, '< 6 GB (very slow - use Colab if possible)'),
    ]
    print(f'CUDA GPU: {device_name}  ({vram_gb:.1f} GB VRAM)  Compute {cc_major}.{cc_minor}')
    print('Recommended parameters by VRAM:')
    print('-' * 85)
    print(f'{"Model":<26} {"Batch":>5} {"Accum":>5} {"Eff":>5} {"Eval":>5}  {"BF16":>5}  {"FP16":>5}  Tier')
    print('-' * 85)
    for _vmin, _m, _bs, _ga, _eb, _b16, _f16, _tier in _tiers:
        print(f'{_m:<26} {_bs:>5} {_ga:>5} {_bs*_ga:>5} {_eb:>5}  {str(_b16):>5}  {str(_f16):>5}  {_tier}')
    print('-' * 85)
elif DEVICE_TYPE == "mps":
    print('Using Apple MPS. BF16/FP16, fused AdamW, and CUDA memory stats are disabled.')
else:
    print('Using CPU. This is useful for smoke tests only; full fine-tuning will be very slow.')
print(f'Current config: PER_GPU_BATCH={BATCH_SIZE} ACCUM={GRAD_ACCUM} DEVICES={TRAIN_DEVICE_COUNT} GLOBAL_EFF={GLOBAL_EFFECTIVE_BATCH} BF16={BF16} FP16={FP16} OPTIM={OPTIMIZER_NAME} MODEL={MODEL_NAME}')
print(f'Trainer config: workers={DATALOADER_NUM_WORKERS} logging={LOGGING_STEPS} early_stop={EARLY_STOPPING_PATIENCE} save_limit={SAVE_TOTAL_LIMIT}')
print(f'Calibration  : T={TEMPERATURE_SEARCH_BOUNDS} threshold=[{THRESHOLD_SEARCH_MIN}, {THRESHOLD_SEARCH_MAX}] x {THRESHOLD_SEARCH_STEPS} x {THRESHOLD_SEARCH_PASSES} passes')

# -- Label map -----------------------------------------------------------
LABEL_MAP = {
    'ban-doc':               'Bạn đọc',
    'bao-ve-nguoi-tieu-dung':'Bảo vệ người tiêu dùng',
    'bat-dong-san':          'Bất động sản',
    'chinh-tri':             'Chính trị',
    'cong-nghe':             'Công nghệ',
    'dan-toc-ton-giao':      'Dân tộc - Tôn giáo',
    'doi-song':              'Đời sống',
    'du-lich':               'Du lịch',
    'giao-duc':              'Giáo dục',
    'kinh-doanh':            'Kinh doanh',
    'oto-xe-may':            'Ô tô - Xe máy',
    'phap-luat':             'Pháp luật',
    'suc-khoe':              'Sức khỏe',
    'the-gioi':              'Thế giới',
    'the-thao':              'Thể thao',
    'thi-truong-tieu-dung':  'Thị trường tiêu dùng',
    'thoi-su':               'Thời sự',
    'tuan-viet-nam':         'Tuần Việt Nam',
    'van-hoa-giai-tri':      'Văn hóa - Giải trí',
}

# -- Helpers -------------------------------------------------------------
_T0 = time.time()

def log(msg, level='INFO'):
    icons = {'INFO':'i','OK':'OK','WARN':'!!','SAVE':'>>','GPU':'GPU'}
    elapsed = time.time() - _T0
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[{ts}][{elapsed:6.1f}s] {icons.get(level," ")} {msg}', flush=True)

class timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *_): log(f'{self.label} - {time.time()-self.t:.1f}s', 'OK')

def save_fig(fig, filename):
    path = RESULTS_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches='tight')
    log(f'Saved -> {path}', 'SAVE')
    plt.show(); plt.close(fig)

def release_accelerator_cache():
    if DEVICE_TYPE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE_TYPE == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()

def gpu_status():
    if DEVICE_TYPE == "cuda":
        used = torch.cuda.memory_allocated(0) / 1024**3
        resv = torch.cuda.memory_reserved(0)  / 1024**3
        log(f'VRAM: {used:.1f}GB allocated / {resv:.1f}GB reserved / {vram_gb-resv:.1f}GB free', 'GPU')
    elif DEVICE_TYPE == "mps":
        if hasattr(torch, "mps") and hasattr(torch.mps, "current_allocated_memory"):
            used = torch.mps.current_allocated_memory() / 1024**3
            log(f'MPS memory: {used:.2f}GB allocated', 'GPU')
        else:
            log('MPS memory stats are not available in this PyTorch build', 'GPU')
    else:
        log('Runtime device: CPU', 'GPU')

# -- Dataset status ------------------------------------------------------
_pq = [f.name for f in DATASET_FOLDER.glob('*.parquet')] if DATASET_FOLDER.exists() else []
print()
print(f'Repo root     : {REPO_ROOT}')
print(f'Dataset folder: {DATASET_FOLDER}')
print(f'URL dataset   : {DATA_URLS_PATH if DATA_URLS_PATH.exists() else "missing"}')
print(f'Config OK     |  {len(LABEL_MAP)} classes  |  {len(_pq)} parquet files')
print(f'  Model    : {MODEL_NAME}')
print(f'  MAX_LEN  : {MAX_LENGTH}  |  Strategy: Head-Tail (127+127)')
print(f'  Batch    : {BATCH_SIZE} x accum {GRAD_ACCUM} x devices {TRAIN_DEVICE_COUNT} = eff {GLOBAL_EFFECTIVE_BATCH}  |  Eval: {EVAL_BATCH}')
print(f'  LR       : {LR}  |  Epochs: {NUM_EPOCHS}  |  BF16: {BF16}  |  FP16: {FP16}')
print(f'  Optim    : {OPTIMIZER_NAME}  |  Pin memory: {PIN_MEMORY}')
print()
print('  Cache:')
for _n, _p in [('temp/processed_data.pkl', PROCESSED_DATA_PATH),
               ('temp/processed_data.meta.json', PROCESSED_META_PATH),
               ('temp/headtail_train.pkl', HEADTAIL_TRAIN_CACHE_PATH),
               ('temp/headtail_test.pkl', HEADTAIL_TEST_CACHE_PATH),
               ('model/config.json', MODEL_CONFIG_PATH),
               ('model/label_config.json', LABEL_CONFIG_PATH),
               ('model/thresholds.json', THRESHOLD_PATH),
               ('model/train_history.pkl', TRAIN_HISTORY_PATH)]:
    print(f'    {_n:<30}: {"yes" if os.path.exists(_p) else "no"}')





In [ ]:
# -- 0.4 Validate or materialize the dataset -----------------------------
import importlib.util
import pyarrow.parquet as _pq_check

_ok = True
_errors = []
_warnings = []
_expected = set(LABEL_MAP.keys())

if not DATASET_FOLDER.is_dir():
    DATASET_FOLDER.mkdir(parents=True, exist_ok=True)

def _dataset_has_local_inputs():
    return any(DATASET_FOLDER.glob("*.parquet")) or DATA_URLS_PATH.exists()

if AUTO_DOWNLOAD_DATASET and not _dataset_has_local_inputs():
    try:
        from huggingface_hub import snapshot_download
        print(f"Downloading dataset from Hugging Face: {HF_DATASET_REPO}")
        snapshot_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            local_dir=str(DATASET_FOLDER),
            allow_patterns=["*.parquet", "*.json", "README.md"],
        )
        print(f"Downloaded dataset to {DATASET_FOLDER}")
    except Exception as _e:
        _warnings.append(f"[!] Could not download {HF_DATASET_REPO}: {_e}")

_pq_files = sorted(f.name for f in DATASET_FOLDER.glob('*.parquet'))
_found    = {f.replace('.parquet', '') for f in _pq_files}
_missing  = sorted(_expected - _found)

if _missing and AUTO_MATERIALIZE_DATASET:
    if DATA_URLS_PATH.exists():
        print(f"Missing {len(_missing)} parquet files. Materializing from {DATA_URLS_PATH.name}...")
        print(f"  Workers={CRAWL_WORKERS} | Batch={CRAWL_BATCH_SIZE} | Limit/category={URLS_PER_CATEGORY or 'full'}")
        _script_path = SCRIPT_DIR / "materialize_vietnamnet_dataset.py"
        _spec = importlib.util.spec_from_file_location("materialize_vietnamnet_dataset", _script_path)
        _module = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_module)
        _module.materialize_dataset(
            DATASET_FOLDER,
            categories=LABEL_MAP.keys(),
            workers=CRAWL_WORKERS,
            batch_size=CRAWL_BATCH_SIZE,
            max_urls_per_category=URLS_PER_CATEGORY,
            force=False,
        )
        _pq_files = sorted(f.name for f in DATASET_FOLDER.glob('*.parquet'))
        _found    = {f.replace('.parquet', '') for f in _pq_files}
        _missing  = sorted(_expected - _found)
    else:
        _warnings.append(f"[!] Missing {DATA_URLS_PATH}; cannot materialize parquet files from the upstream URL list")

if _missing:
    _errors.append(f"[ERROR] Missing {len(_missing)} parquet files: {_missing}")
    _ok = False
if len(_found) != len(_expected):
    _errors.append(f"[ERROR] Expected {len(_expected)} parquet files, found {len(_found)}")
    _ok = False

# Each expected parquet file must be readable and non-empty.
if _ok:
    for _f in _pq_files:
        _name = _f.replace('.parquet', '')
        if _name not in _expected:
            continue
        _path = DATASET_FOLDER / _f
        try:
            _meta = _pq_check.read_metadata(_path)
            if _meta.num_rows == 0:
                _errors.append(f"[ERROR] Empty parquet file: {_f}")
                _ok = False
        except Exception as _e:
            _errors.append(f"[ERROR] Could not read {_f}: {_e}")
            _ok = False

if DEVICE_TYPE == "cuda":
    print(f"[OK] CUDA device: {device_name}  ({vram_gb:.1f} GB VRAM)")
    if vram_gb < 8:
        _warnings.append(f"[!] VRAM is only {vram_gb:.1f} GB; reduce batch size if training is unstable")
elif DEVICE_TYPE == "mps":
    print("[OK] MPS device: Apple Metal Performance Shaders")
    _warnings.append("[!] MPS training is supported but slower and less mature than CUDA")
else:
    print("[OK] CPU runtime selected")
    _warnings.append("[!] CPU training is supported for smoke tests but will be extremely slow for full PhoBERT fine-tuning")

if _ok:
    _rows = []
    for _f in sorted(_pq_files):
        _name = _f.replace('.parquet', '')
        if _name not in _expected:
            continue
        _meta = _pq_check.read_metadata(DATASET_FOLDER / _f)
        _rows.append((_f, _meta.num_rows))
    print(f"[OK] Dataset ready - {len(_rows)} parquet files")
    for _f, _n in _rows:
        print(f"  {_f:<36} {_n:>8,} rows")

for _w in _warnings:
    print(_w)

if not _ok:
    print("\n" + "="*72)
    print("  CANNOT CONTINUE - Dataset is not ready")
    print("="*72)
    for _e in _errors:
        print(f"  {_e}")
    print("\n  The upstream GitHub repo ships URL lists, not parquet article files.")
    print("  This notebook first tries to download the dataset from Hugging Face.")
    print("  If that is disabled or unavailable, it can materialize parquet files when data_URLs.json exists.")
    print("  Manual command:")
    print(f"  python train/scripts/materialize_vietnamnet_dataset.py --dataset-dir {DATASET_FOLDER}")
    print("="*72 + "\n")
    raise SystemExit("Dataset is not ready. See the message above.")

---
## Section 1 - Load Raw Data
Read all **19 parquet files** from the dataset directory in small batches, keeping only compact training text in memory.

In [ ]:
# -- 1.1 Load raw data in parquet batches --------------------------------
import gc
import pyarrow.parquet as pq

log(f"Reading parquet files from {DATASET_FOLDER} in {PARQUET_BATCH_SIZE:,}-row batches...")
log(f"Rows/class cap: {_rows_cap_label} | Content kept: {RAW_CONTENT_HEAD_WORDS}+{RAW_CONTENT_TAIL_WORDS} words", "INFO")

def _word_count_series(s):
    return s.fillna("").astype(str).str.count(r"\S+").fillna(0).astype("int32")

def _compact_content(value):
    words = str(value or "").split()
    if RAW_CONTENT_HEAD_WORDS <= 0 and RAW_CONTENT_TAIL_WORDS <= 0:
        return " ".join(words)
    keep = max(RAW_CONTENT_HEAD_WORDS, 0) + max(RAW_CONTENT_TAIL_WORDS, 0)
    if keep <= 0 or len(words) <= keep:
        return " ".join(words)
    head = words[:max(RAW_CONTENT_HEAD_WORDS, 0)]
    tail = words[-RAW_CONTENT_TAIL_WORDS:] if RAW_CONTENT_TAIL_WORDS > 0 else []
    return " ".join(head + tail)

_records = []
_load_stats = []
_missing_title_total = 0
_missing_content_total = 0
_missing_both_total = 0

for _fname in sorted(os.listdir(DATASET_FOLDER)):
    if not _fname.endswith(".parquet"):
        continue
    _slug = _fname.replace(".parquet", "")
    _lbl = LABEL_MAP.get(_slug)
    if _lbl is None:
        continue

    _path = DATASET_FOLDER / _fname
    _pf = pq.ParquetFile(_path)
    _source_rows = _pf.metadata.num_rows
    _kept = 0
    _dropped = 0
    _miss_t = 0
    _miss_c = 0
    _miss_both = 0

    for _batch in _pf.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=["title", "content"]):
        if MAX_ROWS_PER_CLASS and _kept >= MAX_ROWS_PER_CLASS:
            break

        _dfb = _batch.to_pandas()
        _title = _dfb["title"].fillna("").astype(str).str.strip()
        _content = _dfb["content"].fillna("").astype(str).str.strip()
        _empty_t = _title.eq("")
        _empty_c = _content.eq("")
        _empty_both = _empty_t & _empty_c

        _miss_t += int(_empty_t.sum())
        _miss_c += int(_empty_c.sum())
        _miss_both += int(_empty_both.sum())

        if _empty_both.any():
            _title = _title[~_empty_both]
            _content = _content[~_empty_both]
            _dropped += int(_empty_both.sum())

        if MAX_ROWS_PER_CLASS:
            _remaining = MAX_ROWS_PER_CLASS - _kept
            if _remaining <= 0:
                break
            _title = _title.iloc[:_remaining]
            _content = _content.iloc[:_remaining]

        if len(_title) == 0:
            continue

        _title_words = _word_count_series(_title)
        _content_words = _word_count_series(_content)
        _part = pd.DataFrame({
            "label": _lbl,
            "title": _title.to_numpy(dtype=object),
            "content": _content.map(_compact_content).to_numpy(dtype=object),
            "title_words": _title_words.to_numpy(dtype="int32"),
            "content_words": _content_words.to_numpy(dtype="int32"),
        })
        _part["text_len"] = (_part["title_words"] + _part["content_words"]).astype("int32")
        _records.append(_part)
        _kept += len(_part)

        del _dfb, _title, _content, _title_words, _content_words, _part

    _missing_title_total += _miss_t
    _missing_content_total += _miss_c
    _missing_both_total += _miss_both
    _load_stats.append((_fname, _source_rows, _kept, _dropped, _lbl))
    log(f"  {_fname:<36} source={_source_rows:>7,} kept={_kept:>6,} dropped={_dropped:>4,} [{_lbl}]")
    gc.collect()

if not _records:
    raise SystemExit("No training rows were loaded from parquet files.")

df_raw = pd.concat(_records, ignore_index=True)
del _records
gc.collect()

log(f"Loaded total: {len(df_raw):,} articles | {df_raw['label'].nunique()} classes", "OK")

print(f"\n   [Data quality checks]")
print(f"   Missing title          : {_missing_title_total:,} source rows")
print(f"   Missing content        : {_missing_content_total:,} source rows")
print(f"   Missing both (dropped) : {_missing_both_total:,} source rows")
print(f"   Content stored         : compact head/tail text for training")
print(f"   Raw memory usage       : {df_raw.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\n   Columns : {list(df_raw.columns)}")
print(f"   Final total   : {len(df_raw):,} articles")

---
## Section 2 - Exploratory Data Analysis
Inspect class distribution, text length, and summary statistics.

In [ ]:
# -- 2.1 Class distribution ----------------------------------------------------------
_vc = df_raw["label"].value_counts().sort_values(ascending=False)
_total = int(_vc.sum())
_mean = float(_vc.mean())
_median = float(_vc.median())
_warn_thr = _median * 0.75

_colors = ["#e76f51" if v < _warn_thr else "#6c8ebf" for v in _vc.values]

fig, ax = plt.subplots(figsize=(13, 9))
_bars = ax.barh(_vc.index, _vc.values, color=_colors, edgecolor="white", height=0.72)
ax.invert_yaxis()

if len(_vc) <= 25:
    for bar, n in zip(_bars, _vc.values):
        ax.text(bar.get_width() + _total * 0.003,
                bar.get_y() + bar.get_height()/2,
                f"{n:,}", va="center", fontsize=9)

ax.axvline(_mean, color="#1f77b4", ls="--", lw=1.8, label=f"Mean = {_mean:,.0f}")
ax.axvline(_median, color="#2ca02c", ls=":", lw=2.0, label=f"Median = {_median:,.0f}")
ax.set_xlim(0, _vc.max() * 1.22)
ax.set_xlabel("Article count")
ax.set_ylabel("Class")
ax.set_title("Some classes have substantially fewer samples", fontweight="bold")
ax.grid(axis="x", alpha=0.25)
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
save_fig(fig, "01_class_distribution.png")

_ir = _vc.max() / _vc.min()
print(f"  Largest class : {_vc.idxmax():<35} {_vc.max():>8,}")
print(f"  Smallest class    : {_vc.idxmin():<35} {_vc.min():>8,}")
print(f"  Imbalance  : {_ir:.2f}x")

In [ ]:
# -- 2.2 Text length ---------------------------------------------------------
_title_len = df_raw["title_words"] if "title_words" in df_raw.columns else df_raw["title"].fillna("").astype(str).str.split().str.len()
_content_len = df_raw["content_words"] if "content_words" in df_raw.columns else df_raw["content"].fillna("").astype(str).str.split().str.len()
_lens = df_raw["text_len"]

fig, axes = plt.subplots(1, 3, figsize=(23, 6))

sns.histplot(_title_len, bins=70, kde=True, color="#6c8ebf", ax=axes[0])
axes[0].set_title("Titles are usually short", fontweight="bold")
axes[0].set_xlabel("Words in title")
axes[0].grid(alpha=0.25)

sns.histplot(_content_len, bins=80, kde=True, color="#2a9d8f", ax=axes[1])
axes[1].axvline(256, color="#e76f51", ls="--", lw=1.8, label="PhoBERT max_length=256")
axes[1].legend(fontsize=8)
axes[1].set_title("Many articles exceed 256 tokens, so Head-Tail helps", fontweight="bold")
axes[1].set_xlabel("Words in content")
axes[1].grid(alpha=0.25)

_cls_med = (df_raw.groupby("label")["text_len"].median().sort_values(ascending=False))
_ord = list(_cls_med.index)
sns.boxplot(data=df_raw, y="label", x="text_len", order=_ord, orient="h",
            ax=axes[2], color="#9ecae1", fliersize=1.6, linewidth=0.9)
axes[2].set_title("Text length median differs by class", fontweight="bold")
axes[2].set_xlabel("Words (title + content)")
axes[2].set_ylabel("")
axes[2].grid(axis="x", alpha=0.25)

fig.tight_layout()
save_fig(fig, "02_text_length.png")

_trunc_pct = (_lens > 256).mean()
print(f"  p50 : {_lens.quantile(0.50):,.1f}")
print(f"  p90 : {_lens.quantile(0.90):,.1f}")
print(f"  p95 : {_lens.quantile(0.95):,.1f}")
print(f"  max : {_lens.max():,.1f}")
print(f"  Tren 256 tu: {_trunc_pct:.1%}")

In [ ]:
# --
_rows = []
for _cls in sorted(df_raw["label"].unique()):
    _sub = df_raw[df_raw["label"] == _cls]["text_len"]
    _rows.append({"Class": _cls, "Articles": len(_sub),
                  "Share %": round(len(_sub)/len(df_raw)*100, 2),
                  "Avg words": round(_sub.mean(), 0), "Median words": round(_sub.median(), 0),
                  "Max words": int(_sub.max())})
_df_sum = pd.DataFrame(_rows).sort_values("Articles", ascending=False).reset_index(drop=True)
_df_sum.index += 1
print("DATASET SUMMARY\n")
display(_df_sum.style
    .background_gradient(subset=["Articles"], cmap="Blues")
    .background_gradient(subset=["Avg words"], cmap="Greens")
    .format({"Articles":"{:,}", "Share %":"{:.2f}%", "Avg words":"{:.0f}", "Median words":"{:.0f}"}))

---
## Section 3 - Text Preprocessing
**PhoBERT note**: do not remove stopwords; BERT models learn context from the full sequence.

Pipeline: lowercase -> remove punctuation -> remove digits -> ViTokenizer.

- Cell 3.1 tokenizes the corpus and caches `temp/processed_data.pkl`.
- Cell 3.2 always reloads the processed data.

In [ ]:
# -- 3.1 Tokenize and cache in batches -----------------------------------
_cache_valid = False
if os.path.exists(PROCESSED_DATA_PATH):
    try:
        with open(PROCESSED_META_PATH, "r", encoding="utf-8") as _f:
            _cache_valid = json.load(_f) == PROCESSED_CACHE_META
    except Exception:
        _cache_valid = False
    if _cache_valid:
        print(f"[OK] Cache exists: {PROCESSED_DATA_PATH} - skipping tokenization")
    else:
        print(f"[INFO] Cache exists but config changed; rebuilding: {PROCESSED_DATA_PATH}")
        for _cache_file in [PROCESSED_DATA_PATH, PROCESSED_META_PATH]:
            if os.path.exists(_cache_file):
                os.remove(_cache_file)

if not _cache_valid:
    log("Starting batched text preprocessing...")

    def _clean_one(text):
        '''ViTokenizer without stopword removal because PhoBERT learns context.'''
        from pyvi import ViTokenizer
        import re
        if not isinstance(text, str): return ""
        text = text.lower()
        text = re.sub(r"[^\w\s]", " ", text)
        text = re.sub(r"\d+",     " ", text)
        text = ViTokenizer.tokenize(text)
        return re.sub(r"\s+", " ", text).strip()

    from joblib import Parallel, delayed
    from multiprocessing import cpu_count
    _jobs = PREPROCESS_N_JOBS if PREPROCESS_N_JOBS > 0 else max(cpu_count() - 1, 1)
    log(f"Tokenizing {len(df_raw):,} articles | batch={PREPROCESS_BATCH_SIZE:,} | jobs={_jobs}")

    _parts = []
    _valid_total = 0
    for _start in range(0, len(df_raw), PREPROCESS_BATCH_SIZE):
        _chunk = df_raw.iloc[_start:_start + PREPROCESS_BATCH_SIZE]
        _texts = (_chunk["title"].astype(str) + " " + _chunk["content"].astype(str)).tolist()
        _labels = _chunk["label"].astype(str).tolist()

        _cleaned = Parallel(n_jobs=_jobs, backend="loky", verbose=0)(
            delayed(_clean_one)(_t) for _t in _texts
        )
        _part = pd.DataFrame({"label": _labels, "clean_text": _cleaned})
        _part = _part[_part["clean_text"].str.strip() != ""]
        _valid_total += len(_part)
        _parts.append(_part)

        if (_start // PREPROCESS_BATCH_SIZE + 1) % 10 == 0 or _start + PREPROCESS_BATCH_SIZE >= len(df_raw):
            log(f"  processed {min(_start + PREPROCESS_BATCH_SIZE, len(df_raw)):>7,}/{len(df_raw):,} rows | valid={_valid_total:,}")

        del _chunk, _texts, _labels, _cleaned, _part
        gc.collect()

    _df_proc = pd.concat(_parts, ignore_index=True)
    del _parts
    gc.collect()
    log(f"After tokenization: {len(_df_proc):,} articles", "OK")

    _classes  = sorted(_df_proc["label"].unique().tolist())
    _label2id = {l: i for i, l in enumerate(_classes)}
    _id2label = {i: l for l, i in _label2id.items()}
    _df_proc["label_id"] = _df_proc["label"].map(_label2id)

    with timer("Save processed_data.pkl"):
        with open(PROCESSED_DATA_PATH, "wb") as _f:
            pickle.dump({"df": _df_proc, "label2id": _label2id, "id2label": _id2label}, _f)
    with open(PROCESSED_META_PATH, "w", encoding="utf-8") as _f:
        json.dump(PROCESSED_CACHE_META, _f, ensure_ascii=False, indent=2)
    log(f"Saved ({os.path.getsize(PROCESSED_DATA_PATH)/1e6:.1f} MB)", "SAVE")


In [ ]:
# -- 3.2 Load processed data ---------------------------------------------
with timer("Load processed_data.pkl"):
    with open(PROCESSED_DATA_PATH, "rb") as f:
        _s = pickle.load(f)

df       = _s["df"]
label2id = _s["label2id"]
id2label = _s["id2label"]
classes  = sorted(df["label"].unique().tolist())
N_CLASSES = len(classes)

_vc3 = df["label"].value_counts()
print(f"[OK] Processed data: {len(df):,} articles | {N_CLASSES} classes\n")
print(f"   {'Class':<38}  {'Articles':>8}  {'%':>5}  TB tokens (ViTokenizer)")
print(f"   {'-'*62}")
for _cls in classes:
    _n   = _vc3.get(_cls, 0)
    _tl  = df[df["label"]==_cls]["clean_text"].str.split().str.len().mean()
    print(f"   {_cls:<38}  {_n:>8,}  {_n/len(df)*100:>4.1f}%  {_tl:>6.0f}")

---
## Section 4 - Dataset and Tokenization
**Head-Tail strategy**: keep the first 127 tokens and the last 127 tokens to fit a 256-token sequence including `[CLS]` and `[SEP]`.

This works well for long news articles because it keeps both the lead and the conclusion.

In [ ]:
# --
class HeadTailDataset(TorchDataset):
    '''Head+Tail: first 127 tokens + last 127 tokens (+ [CLS] + [SEP] = 256).'''
    def __init__(self, texts, labels, tokenizer, label2id):
        _texts = texts.tolist() if hasattr(texts, "tolist") else list(texts)
        _label_ids = [label2id[l] for l in labels]
        self.input_ids  = torch.empty((len(_label_ids), MAX_LENGTH), dtype=torch.long)
        self.attn_masks = torch.empty((len(_label_ids), MAX_LENGTH), dtype=torch.long)
        self.labels     = torch.tensor(_label_ids, dtype=torch.long)
        half   = (MAX_LENGTH - 2) // 2
        cls_id = tokenizer.cls_token_id
        sep_id = tokenizer.sep_token_id
        pad_id = tokenizer.pad_token_id

        for idx, text in enumerate(tqdm(_texts, desc="  HeadTail tokenize", ncols=80, leave=True)):
            tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)
            if len(tokens) <= MAX_LENGTH - 2:
                enc  = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LENGTH)
                ids  = enc["input_ids"]
                attn = enc["attention_mask"]
            else:
                head  = tokens[:half]
                tail  = tokens[-half:]
                ids   = [cls_id] + head + tail + [sep_id]
                attn  = [1] * len(ids)
                pad_n = MAX_LENGTH - len(ids)
                ids  += [pad_id] * pad_n
                attn += [0]     * pad_n
            self.input_ids[idx] = torch.tensor(ids, dtype=torch.long)
            self.attn_masks[idx] = torch.tensor(attn, dtype=torch.long)

    def _to_tensors(self):
        '''Convert Python lists to stacked tensors. Call after loading an older cache format.'''
        if isinstance(self.input_ids, list):
            self.input_ids  = torch.tensor(self.input_ids,  dtype=torch.long)
            self.attn_masks = torch.tensor(self.attn_masks, dtype=torch.long)
            self.labels     = torch.tensor(self.labels,     dtype=torch.long)
        elif not isinstance(self.labels, torch.Tensor):
            self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attn_masks[idx],
            "labels":         self.labels[idx],
        }


class WeightedTrainer(Trainer):
    '''Trainer with weighted cross-entropy for class imbalance.'''
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        weight  = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss    = torch.nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


class EpochCallback(TrainerCallback):
    '''Log details after each epoch and elapsed time every logging step.'''
    def __init__(self):
        self._train_start  = None
        self._epoch_start  = None
        self._total_steps  = None

    def on_train_begin(self, args, state, control, **kwargs):
        self._train_start = time.time()
        self._total_steps = state.max_steps
        log(f"Total steps: {self._total_steps:,}  |  {NUM_EPOCHS} epochs  |  "
            f"~{self._total_steps//NUM_EPOCHS:,} steps/epoch", "OK")

    def on_epoch_begin(self, args, state, control, **kwargs):
        self._epoch_start = time.time()
        ep = int(state.epoch) + 1
        elapsed = time.time() - self._train_start if self._train_start else 0
        log(f"{'-'*50}")
        log(f"EPOCH {ep}/{NUM_EPOCHS} - start  |  Elapsed: {elapsed/60:.1f} min")
        gpu_status()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs: return
        step    = state.global_step
        total   = self._total_steps or 1
        elapsed = time.time() - self._train_start if self._train_start else 0
        eta_sec = (elapsed / step * (total - step)) if step > 0 else 0
        spd     = logs.get("train_samples_per_second", 0)
        spd_str = f"  |  {spd:.0f} samples/s" if spd else ""
        log(f"step {step:>5}/{total}  [{step/total*100:4.1f}%]  "
            f"loss={logs['loss']:.4f}  lr={logs.get('learning_rate',0):.2e}  "
            f"elapsed={elapsed/60:.1f}m  ETA={eta_sec/60:.1f}m{spd_str}")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics: return
        acc   = metrics.get("eval_accuracy",    0)
        f1_w  = metrics.get("eval_f1_weighted", 0)
        f1_m  = metrics.get("eval_f1_macro",    0)
        eloss = metrics.get("eval_loss",        0)
        ep_t  = time.time() - self._epoch_start if self._epoch_start else 0
        total_t = time.time() - self._train_start if self._train_start else 0
        log(f"EPOCH {state.epoch:.0f} DONE  |  Loss={eloss:.4f}  Acc={acc:.4f}  "
            f"F1-w={f1_w:.4f}  F1-m={f1_m:.4f}  "
            f"(epoch {ep_t/60:.1f}m  /  total {total_t/60:.1f}m)", "OK")

print("[OK] Dataset classes and WeightedTrainer are ready.")

In [ ]:
# --
SMOKE_EXAMPLES = df_raw[["title", "content", "label"]].sample(
    min(SMOKE_SAMPLE_SIZE, len(df_raw)), random_state=RANDOM_STATE
).to_dict(orient="records")

log("Chia train/test (stratified)...")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label"]
)
log(f"Train: {len(X_train_raw):,}  |  Test: {len(X_test_raw):,}", "OK")

# Loading tokenizer PhoBERT
log(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --
_sample_text = df["clean_text"].sample(min(TOKEN_LENGTH_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).tolist()
_tlen = np.array([len(tokenizer.encode(t, truncation=False)) for t in
                  tqdm(_sample_text, desc="  Token length check", ncols=80)])
log(f"Token length - mean={_tlen.mean():.1f} | median={np.median(_tlen):.1f} | "
    f"max={_tlen.max()} | >256: {(_tlen>256).mean():.1%}", "OK")

# Computing class weights
log("Computing class weights (balanced + manual boost)...")
_cw_vals = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(N_CLASSES),
    y=[label2id[l] for l in y_train],
)
for i, cls in enumerate(classes):
    _cw_vals[i] *= MANUAL_CLASS_BOOST.get(cls, 1.0)
class_weights = torch.tensor(_cw_vals, dtype=torch.float32)
print(f"\n   {'Class':<38}  {'Weight':>8}  {'Boost':>8}")
print(f"   {'-'*60}")
for i, cls in enumerate(classes):
    _boost = MANUAL_CLASS_BOOST.get(cls, 1.0)
    print(f"   {cls:<38}  {_cw_vals[i]:>8.4f}  {_boost:>8.2f}")

# --
if os.path.exists(HEADTAIL_TRAIN_CACHE_PATH) and os.path.exists(HEADTAIL_TEST_CACHE_PATH):
    log("Loading HeadTail datasets from cache...", "OK")
    with timer("Load train cache"):
        with open(HEADTAIL_TRAIN_CACHE_PATH, "rb") as f:
            train_dataset = pickle.load(f)
    with timer("Load test cache"):
        with open(HEADTAIL_TEST_CACHE_PATH, "rb") as f:
            test_dataset = pickle.load(f)
    # Older cache stores lists, so convert to stacked tensors for faster __getitem__
    train_dataset._to_tensors()
    test_dataset._to_tensors()
    log("Loaded HeadTail datasets from cache", "OK")
else:
    log("Creating HeadTail train dataset for the first time; it will be cached...")
    with timer("Create HeadTail train"):
        train_dataset = HeadTailDataset(X_train_raw, y_train, tokenizer, label2id)
    with timer("Save train cache"):
        with open(HEADTAIL_TRAIN_CACHE_PATH, "wb") as f:
            pickle.dump(train_dataset, f)
    log(f"Saved cache train ({os.path.getsize(HEADTAIL_TRAIN_CACHE_PATH)/1e6:.0f} MB)", "SAVE")

    log("Creating HeadTail test dataset...")
    with timer("Create HeadTail test"):
        test_dataset = HeadTailDataset(X_test_raw, y_test, tokenizer, label2id)
    with timer("Save test cache"):
        with open(HEADTAIL_TEST_CACHE_PATH, "wb") as f:
            pickle.dump(test_dataset, f)
    log(f"Saved cache test ({os.path.getsize(HEADTAIL_TEST_CACHE_PATH)/1e6:.0f} MB)", "SAVE")

del X_train_raw, X_test_raw, df, df_raw
gc.collect()
log("Released raw text DataFrames; training now uses tokenized dataset caches", "OK")

log(f"Train: {len(train_dataset):,} samples  |  Test: {len(test_dataset):,} samples", "OK")
log(f"input_ids shape: {train_dataset.input_ids.shape}  dtype={train_dataset.input_ids.dtype}", "OK")
log(f"[SAVE] Delete cache when no longer needed: {HEADTAIL_TRAIN_CACHE_PATH}", "INFO")
gpu_status()


---
## Section 5 - Model Training
Fine-tune **PhoBERT Base v2** with Hugging Face Trainer.

| Parameter | Value | Reason |
|-----------|-------|--------|
| `LR` | 2e-5 | Avoid catastrophic forgetting on a pretrained model |
| `batch` | 32 with accum 2 = effective 64 | Fits a 12GB GPU while keeping effective batch size 64 |
| `class_weight` | balanced | Compensates for class imbalance |
| `BF16` | True | Supported by Ampere GPUs such as RTX A3000 |
| `metric_for_best_model` | f1_macro | Prioritizes smaller classes more than weighted F1 |

- Cell 5.1 trains the model, unless `model/config.json` already exists.
- Cell 5.2 always loads the model and evaluates on the test set.

In [ ]:
# -- 5.1 Train PhoBERT and save the model ---------------------------------
_MODEL_READY = os.path.exists(MODEL_CONFIG_PATH)

if _MODEL_READY:
    log(f"Model already exists: {MODEL_DIR} - skipping training", "OK")
else:
    release_accelerator_cache()
    log(f"Loading PhoBERT model: {MODEL_NAME}...")

    def make_compute_metrics():
        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=1)
            return {
                "accuracy":    accuracy_score(labels, preds),
                "f1_weighted": f1_score(labels, preds, average="weighted"),
                "f1_macro":    f1_score(labels, preds, average="macro"),
            }
        return compute_metrics

    model_train = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES,
        id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)

    # Normalize LayerNorm beta/gamma keys from older PhoBERT checkpoints
    _sd = model_train.state_dict()
    _new_sd = {}
    for k, v in _sd.items():
        if k.endswith(".beta"):    _new_sd[k.replace(".beta",  ".bias")]   = v
        elif k.endswith(".gamma"): _new_sd[k.replace(".gamma", ".weight")] = v
        else:                      _new_sd[k] = v
    model_train.load_state_dict(_new_sd, strict=False)

    log(f"Parameters: {sum(p.numel() for p in model_train.parameters())/1e6:.1f}M", "OK")
    gpu_status()

    training_args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,       # 64
        per_device_eval_batch_size=EVAL_BATCH,        # 128
        gradient_accumulation_steps=GRAD_ACCUM,       # 1
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        **EVAL_STRATEGY_KWARG,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        bf16=BF16,
        fp16=FP16,
        logging_steps=LOGGING_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        # num_workers=0: required on Windows (Python 3.14 spawn multiprocessing
        # can crash with large tensors in workers). The dataset already uses stacked tensors
        # so __getitem__ is fast; the GPU is the bottleneck.
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=PIN_MEMORY,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,                 # OFF; base-v2 fits in 12GB
        optim=OPTIMIZER_NAME,                         # fused AdamW only on CUDA
        torch_compile=TORCH_COMPILE,                          # Python 3.14+ is not supported yet
    )

    trainer = WeightedTrainer(
        model=model_train,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=make_compute_metrics(),
        class_weights=class_weights,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
            EpochCallback(),
        ],
    )

    log("Starting PhoBERT fine-tuning...")
    _t_start = time.time()
    trainer.train()
    _train_time = time.time() - _t_start
    log(f"Training finished - {_train_time:.0f}s ({_train_time/60:.1f} min)", "OK")
    gpu_status()

    # Save model and tokenizer
    trainer.save_model(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    log(f"Saved model: {MODEL_DIR}", "SAVE")

    # Save training history
    with open(TRAIN_HISTORY_PATH, "wb") as f:
        pickle.dump({"log_history": trainer.state.log_history,
                     "train_time_sec": _train_time}, f)
    log(f"Saved training history: {TRAIN_HISTORY_PATH}", "SAVE")

    # Clean checkpoints
    if os.path.exists(CHECKPOINT_DIR):
        shutil.rmtree(CHECKPOINT_DIR)
    del model_train, trainer
    release_accelerator_cache()
    log("Accelerator cache released", "OK")



In [ ]:
# -- 5.2 Load model and predict the test set ----------------------------------
log(f"Loading model from {MODEL_DIR}...")
model_eval = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=N_CLASSES,
    id2label=id2label,
    label2id=label2id,
).to(device)
model_eval.eval()

def make_compute_metrics():
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {"accuracy":    accuracy_score(labels, preds),
                "f1_weighted": f1_score(labels, preds, average="weighted"),
                "f1_macro":    f1_score(labels, preds, average="macro")}
    return compute_metrics

_eval_args = TrainingArguments(
    output_dir=TEMP_DIR, per_device_eval_batch_size=EVAL_BATCH,
    bf16=BF16, fp16=FP16, report_to="none",
    dataloader_num_workers=DATALOADER_NUM_WORKERS,       # Windows: spawn multiprocessing can crash with num_workers > 0
    dataloader_pin_memory=PIN_MEMORY,
)
_eval_trainer = Trainer(
    model=model_eval, args=_eval_args,
    eval_dataset=test_dataset, compute_metrics=make_compute_metrics(),
)

log("Predicting the test set...")
with timer("Inference"):
    _raw = _eval_trainer.predict(test_dataset)

y_pred_ids  = np.argmax(_raw.predictions, axis=1).tolist()
y_true_ids  = [label2id[l] for l in y_test]
raw_logits  = _raw.predictions

model_acc   = accuracy_score(y_true_ids, y_pred_ids)
model_f1w   = f1_score(y_true_ids, y_pred_ids, average="weighted")
model_f1m   = f1_score(y_true_ids, y_pred_ids, average="macro")
f1_per_class= dict(zip(classes, f1_score(y_true_ids, y_pred_ids,
                                          labels=list(range(N_CLASSES)), average=None)))

# Load training history if available
log_history = []
train_time_sec = 0
if os.path.exists(TRAIN_HISTORY_PATH):
    with open(TRAIN_HISTORY_PATH, "rb") as f:
        _hist = pickle.load(f)
    log_history    = _hist.get("log_history", [])
    train_time_sec = _hist.get("train_time_sec", 0)

print(f"\n{'='*55}")
print(f"  PhoBERT Base v2 - Head-Tail")
print(f"  {'-'*52}")
print(f"  Accuracy   : {model_acc:.4f}  ({model_acc*100:.2f}%)")
print(f"  F1-weighted: {model_f1w:.4f}")
print(f"  F1-macro   : {model_f1m:.4f}")
if train_time_sec:
    print(f"  Train time : {train_time_sec/60:.1f} min")
print(f"{'='*55}")
gpu_status()


---
## Section 6 - Model Evaluation
Generate a classification report, confusion matrix, per-class F1, and training curves.

In [ ]:
# -- 6.1 Classification report -----------------------------------------------
_report_str = classification_report(y_true_ids, y_pred_ids, target_names=classes, digits=4)

print(f"{'='*65}")
print(f"  CLASSIFICATION REPORT - PhoBERT Base v2 Head-Tail")
print(f"  {'-'*62}")
print(f"  Accuracy   : {model_acc:.4f}")
print(f"  F1-weighted: {model_f1w:.4f}")
print(f"  F1-macro   : {model_f1m:.4f}")
print(f"{'='*65}")
print(_report_str)

_rp = os.path.join(RESULTS_DIR, "classification_report.txt")
with open(_rp, "w", encoding="utf-8") as f:
    f.write(f"PhoBERT Base v2 - Head-Tail\n")
    f.write(f"Accuracy:    {model_acc:.4f}\n")
    f.write(f"F1-weighted: {model_f1w:.4f}\n")
    f.write(f"F1-macro:    {model_f1m:.4f}\n\n")
    f.write(_report_str)
log(f"Saved: {_rp}", "SAVE")

In [ ]:
# -- 6.2 Confusion matrix + top confusion pairs --------------------------------
_cm = confusion_matrix(y_true_ids, y_pred_ids, labels=list(range(N_CLASSES)))
_cmn = _cm.astype(float) / np.clip(_cm.sum(axis=1, keepdims=True), 1, None)

_pairs = []
for i, t in enumerate(classes):
    for j, p in enumerate(classes):
        if i == j:
            continue
        c = int(_cm[i, j])
        if c > 0:
            _pairs.append((f"{t} -> {p}", c))
_top_pairs = sorted(_pairs, key=lambda x: x[1], reverse=True)[:10]

fig, axes = plt.subplots(1, 3, figsize=(33, 10), gridspec_kw={"width_ratios": [1.3, 1.3, 1.0]})

sns.heatmap(_cmn, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=classes, yticklabels=classes,
            linewidths=0.25, linecolor="white", ax=axes[0], annot_kws={"size": 7})
axes[0].set_title("Normalized view for quick error reading", fontweight="bold")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(axes[0].get_yticklabels(), rotation=0, fontsize=8)

sns.heatmap(_cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=classes, yticklabels=classes,
            linewidths=0.25, linecolor="white", ax=axes[1], annot_kws={"size": 7})
axes[1].set_title("Raw counts show error scale", fontweight="bold")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(axes[1].get_yticklabels(), rotation=0, fontsize=8)

if _top_pairs:
    _labels = [x[0] for x in _top_pairs][::-1]
    _vals = [x[1] for x in _top_pairs][::-1]
    _bars = axes[2].barh(_labels, _vals, color="#e76f51", edgecolor="white")
    for b, v in zip(_bars, _vals):
        axes[2].text(v + max(_vals) * 0.02, b.get_y() + b.get_height()/2, f"{v}", va="center", fontsize=9)
    axes[2].set_title("Top confusion pairs", fontweight="bold")
    axes[2].set_xlabel("Mistake count")
    axes[2].grid(axis="x", alpha=0.25)
else:
    axes[2].axis("off")

fig.suptitle(f"Confusion Matrix — PhoBERT Base v2 | Accuracy={model_acc:.4f}", fontsize=13, fontweight="bold")
fig.tight_layout()
save_fig(fig, "03_confusion_matrix.png")

In [ ]:
# -- 6.3 P/R/F1 by class + support vs F1 ---------------------------------------
_report_dict = classification_report(
    y_true_ids, y_pred_ids, target_names=classes, digits=4,
    output_dict=True, zero_division=0
)
_diag_rows = []
for cls in classes:
    _diag_rows.append({
        "class": cls,
        "precision": _report_dict[cls]["precision"],
        "recall": _report_dict[cls]["recall"],
        "f1": _report_dict[cls]["f1-score"],
        "support": int(_report_dict[cls]["support"]),
    })
_prf = pd.DataFrame(_diag_rows).sort_values("f1", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 10))
_y = np.arange(len(_prf))
_h = 0.24

ax.barh(_y - _h, _prf["precision"], height=_h, color="#6c8ebf", label="Precision")
ax.barh(_y,       _prf["recall"],    height=_h, color="#2a9d8f", label="Recall")
_bf1 = ax.barh(_y + _h, _prf["f1"],  height=_h,
               color=["#e76f51" if v < 0.80 else "#f4a261" if v < 0.90 else "#2a9d8f" for v in _prf["f1"]],
               label="F1")
for b, v in zip(_bf1, _prf["f1"]):
    ax.text(v + 0.004, b.get_y() + b.get_height()/2, f"{v:.3f}", va="center", fontsize=8)

ax.axvline(model_f1m, color="#1f77b4", ls="--", lw=2, label=f"Macro F1 = {model_f1m:.4f}")
ax.set_yticks(_y)
ax.set_yticklabels(_prf["class"], fontsize=9)
ax.set_xlim(0, 1.08)
ax.set_xlabel("Score")
ax.set_title("Weak classes stand out in Precision-Recall balance", fontweight="bold")
ax.grid(axis="x", alpha=0.25)
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
save_fig(fig, "04_f1_per_class.png")

fig2, ax2 = plt.subplots(figsize=(11, 7))
_sc = ax2.scatter(_prf["support"], _prf["f1"],
                  s=np.clip(_prf["support"] / 8, 30, 260),
                  c=_prf["f1"], cmap="RdYlGn", edgecolor="white", alpha=0.9)
for _, r in _prf.iterrows():
    if r["support"] < _prf["support"].quantile(0.25) or r["f1"] < 0.85:
        ax2.text(r["support"] * 1.01, r["f1"] + 0.003, r["class"], fontsize=8)
ax2.axhline(model_f1m, color="#1f77b4", ls="--", lw=1.5, label=f"Macro F1 = {model_f1m:.4f}")
ax2.set_xlabel("Support (test)")
ax2.set_ylabel("F1")
ax2.set_title("Low-support and low-F1 classes stand out", fontweight="bold")
ax2.grid(alpha=0.25)
ax2.legend(loc="lower right", fontsize=9)
fig2.colorbar(_sc, ax=ax2, label="F1")
fig2.tight_layout()
save_fig(fig2, "07_support_vs_f1.png")

In [ ]:
# -- 6.4 Training Curves --------------------------------------------------------
if not log_history:
    print("No log_history available because the model was loaded from cache; skipping training curves.")
else:
    _train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
    _eval_logs  = [x for x in log_history if "eval_loss" in x]

    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

    # 1) train/eval loss
    axes[0].plot([x["step"] for x in _train_logs], [x["loss"] for x in _train_logs],
                 alpha=0.45, color="#6c8ebf", lw=1.0, label="Train loss")
    _ax2 = axes[0].twinx()
    _ax2.plot([x["epoch"] for x in _eval_logs], [x["eval_loss"] for x in _eval_logs],
              marker="o", color="#e76f51", lw=2.0, label="Eval loss")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Train loss", color="#6c8ebf")
    _ax2.set_ylabel("Eval loss", color="#e76f51")
    axes[0].set_title("Loss decreases steadily across epochs", fontweight="bold")
    axes[0].grid(alpha=0.25)

    # 2) eval macro F1
    _ep = [x["epoch"] for x in _eval_logs]
    _f1m = [x.get("eval_f1_macro", 0) for x in _eval_logs]
    axes[1].plot(_ep, _f1m, marker="o", color="#2a9d8f", lw=2)
    for x, y in zip(_ep, _f1m):
        axes[1].annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].set_ylim(max(0.7, min(_f1m) - 0.02), 1.0)
    axes[1].set_title("Macro F1 improves and converges", fontweight="bold")
    axes[1].grid(alpha=0.25)

    # 3) eval accuracy
    _ac = [x.get("eval_accuracy", 0) for x in _eval_logs]
    axes[2].plot(_ep, _ac, marker="o", color="#6c8ebf", lw=2)
    for x, y in zip(_ep, _ac):
        axes[2].annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Accuracy")
    axes[2].set_ylim(max(0.7, min(_ac) - 0.02), 1.0)
    axes[2].set_title("Accuracy reaches a stable high level", fontweight="bold")
    axes[2].grid(alpha=0.25)

    fig.suptitle(f"PhoBERT Base v2 — Training Curves (train {train_time_sec/60:.1f} phut)",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "05_training_curves.png")

In [ ]:
# -- 6.4 Result summary -------------------------------------------------
print(f"\n{'='*60}")
print(f"  RESULTS - PhoBERT Base v2 Head-Tail")
print(f"{'='*60}")
print(f"\n  [MODEL CONFIG]")
print(f"    Model        : {MODEL_NAME}")
print(f"    Strategy     : Head-Tail (127 + 127 tokens)")
print(f"    MAX_LENGTH   : {MAX_LENGTH}")
print(f"    Batch (eff)  : {BATCH_SIZE} x {GRAD_ACCUM} x {TRAIN_DEVICE_COUNT} = {GLOBAL_EFFECTIVE_BATCH}")
print(f"    LR           : {LR}  |  Epochs: {NUM_EPOCHS}  |  BF16: {BF16}")
print(f"    class_weight : balanced")
print(f"\n  [DATA]")
print(f"    Number of classes    : {N_CLASSES}")
print(f"    Train        : {len(y_train):,} articles  ({(1-TEST_SIZE)*100:.0f}%)")
print(f"    Test         : {len(y_test):,} articles   ({TEST_SIZE*100:.0f}%)")
if train_time_sec:
    print(f"    Train time   : {train_time_sec/60:.1f} min  ({train_time_sec/3600:.2f} hours)")
print(f"\n  [TEST PERFORMANCE]")
print(f"    Accuracy     : {model_acc:.4f}  ({model_acc*100:.2f}%)")
print(f"    F1-weighted  : {model_f1w:.4f}")
print(f"    F1-macro     : {model_f1m:.4f}")
print(f"\n  [5 HARDEST CLASSES]")
for _cls, _v in sorted(f1_per_class.items(), key=lambda x: x[1])[:5]:
    print(f"    {_cls:<38}  F1 = {_v:.4f}")
print(f"\n  [5 EASIEST CLASSES]")
for _cls, _v in sorted(f1_per_class.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"    {_cls:<38}  F1 = {_v:.4f}")

_cm2      = confusion_matrix(y_true_ids, y_pred_ids, labels=list(range(N_CLASSES)))
_confused = sorted(
    [(_cm2[i,j], classes[i], classes[j])
     for i in range(N_CLASSES) for j in range(N_CLASSES) if i!=j and _cm2[i,j]>0],
    reverse=True
)
print(f"\n  [TOP 5 CONFUSION PAIRS]")
for _cnt, _tr, _pr in _confused[:5]:
    print(f"    {_tr:<28} -> {_pr:<28}  {_cnt:>4} times")

print(f"\n  [SAVED RESULT FILES]")
for _f in sorted(os.listdir(RESULTS_DIR)):
    _fp = os.path.join(RESULTS_DIR, _f)
    print(f"    {_f:<38}  ({os.path.getsize(_fp)/1024:.1f} KB)")
print(f"\n{'='*60}")



---
## Section 7 - Export Model Config
Save `label_config.json` in `model/` so inference can reuse label and preprocessing metadata.

The model and tokenizer are saved automatically by Trainer in Section 5.

In [ ]:
# --
_label_config = {
    "model_name":  MODEL_NAME,
    "strategy":    "head_tail",
    "max_length":  MAX_LENGTH,
    "n_classes":   N_CLASSES,
    "classes":     classes,
    "label2id":    label2id,
    "id2label":    {str(k): v for k, v in id2label.items()},
    "preprocessing": {
        "title_weight":  1,
        "lowercase":     True,
        "remove_punct":  True,
        "remove_digits": True,
        "tokenizer":     "pyvi.ViTokenizer",
        "stopwords":     False,
    },
}
with open(LABEL_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(_label_config, f, ensure_ascii=False, indent=2)
log(f"Saved: {LABEL_CONFIG_PATH}", "SAVE")

# --
print(f"\n  [SMOKE TEST - 3 RANDOM ARTICLES]\n")

def _infer(title, content):
    from pyvi import ViTokenizer
    text  = (str(title) + " " + str(content)).lower()
    text  = re.sub(r"[^\w\s]", " ", text)
    text  = re.sub(r"\d+",     " ", text)
    text  = ViTokenizer.tokenize(text)
    text  = re.sub(r"\s+",     " ", text).strip()
    half  = (MAX_LENGTH - 2) // 2
    enc_full = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    if len(enc_full) <= MAX_LENGTH - 2:
        enc = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LENGTH,
                        return_tensors="pt")
    else:
        head = enc_full[:half]; tail = enc_full[-half:]
        ids  = [tokenizer.cls_token_id] + head + tail + [tokenizer.sep_token_id]
        ids += [tokenizer.pad_token_id] * (MAX_LENGTH - len(ids))
        enc  = {"input_ids": torch.tensor([ids]),
                "attention_mask": torch.tensor([[1]*min(len(enc_full)+2, MAX_LENGTH)
                                                 + [0]*(MAX_LENGTH - min(len(enc_full)+2, MAX_LENGTH))])}
    with torch.no_grad():
        logits = model_eval(**{k: v.to(device) for k,v in enc.items()}).logits
    return classes[logits.argmax(-1).item()]

_ok  = 0
for _row in SMOKE_EXAMPLES:
    _pred = _infer(_row["title"], _row["content"])
    _ok  += int(_pred == _row["label"])
    _mark = "[OK]" if _pred == _row["label"] else "[FAIL]"
    print(f"  {_mark}  True: {_row['label']:<32}  Pred: {_pred}")
print(f"\n  Smoke test: {_ok}/3 correct")

print(f"\n  [FILES SAVED IN model/]")
for _f in sorted(os.listdir(MODEL_DIR)):
    _fp = os.path.join(MODEL_DIR, _f)
    if os.path.isfile(_fp):
        print(f"    {_f:<35}  ({os.path.getsize(_fp)/1024:.1f} KB)")

---
## Section 8 - Threshold Calibration
Instead of plain `argmax(logits)`, tune one threshold per class to improve F1 for weaker classes.

**Mechanism:** `predict = argmax(probs / thresholds)`. A higher threshold makes a class harder to select; a lower threshold makes it easier to select.

> **Note:** thresholds are tuned on the test set here, so the result is optimistic. Use a separate validation set for production calibration.

In [ ]:
# -- 8.0 Temperature scaling: compute once, reuse if cached --------------------
from scipy.optimize import minimize_scalar
from scipy.special import softmax as sp_softmax
import numpy as _np

_logits = raw_logits
_labels = np.array(y_true_ids)

if os.path.exists(THRESHOLD_PATH):
    with open(THRESHOLD_PATH, "r", encoding="utf-8") as f:
        _thr_cache = json.load(f)
    T_OPTIMAL = float(_thr_cache.get("temperature", 1.0))
    log(f"Threshold cache exists: {THRESHOLD_PATH} -> skipping temperature search", "OK")
else:
    def _nll(T):
        scaled = _logits / max(T, 1e-3)
        p = sp_softmax(scaled, axis=1)
        true_p = p[np.arange(len(_labels)), _labels]
        return -np.mean(np.log(true_p + 1e-10))

    _result = minimize_scalar(_nll, bounds=TEMPERATURE_SEARCH_BOUNDS, method="bounded")
    T_OPTIMAL = float(_result.x)
    log("Temperature scaling complete", "OK")

def _nll_eval(T):
    scaled = _logits / max(T, 1e-3)
    p = sp_softmax(scaled, axis=1)
    true_p = p[np.arange(len(_labels)), _labels]
    return -np.mean(np.log(true_p + 1e-10))

def _ece(logits, labels, T=1.0, n_bins=ECE_N_BINS):
    p = sp_softmax(logits / T, axis=1)
    conf = p.max(axis=1)
    pred = p.argmax(axis=1)
    acc_bins, conf_bins, counts = [], [], []
    for b in range(n_bins):
        lo, hi = b/n_bins, (b+1)/n_bins
        mask = (conf >= lo) & (conf < hi)
        if mask.sum() == 0:
            continue
        acc_bins.append((pred[mask] == labels[mask]).mean())
        conf_bins.append(conf[mask].mean())
        counts.append(mask.sum())
    counts = np.array(counts)
    return np.sum(counts * np.abs(np.array(acc_bins) - np.array(conf_bins))) / counts.sum()

_nll_before = _nll_eval(1.0)
_nll_after  = _nll_eval(T_OPTIMAL)
_ece_before = _ece(_logits, _labels, T=1.0)
_ece_after  = _ece(_logits, _labels, T=T_OPTIMAL)

print(f"  T_optimal = {T_OPTIMAL:.4f}")
print(f"  NLL   : {_nll_before:.4f} -> {_nll_after:.4f}  (delta {_nll_after-_nll_before:+.4f})")
print(f"  ECE   : {_ece_before:.4f} -> {_ece_after:.4f}  (delta {_ece_after-_ece_before:+.4f})")


In [ ]:
# -- 8.1 Threshold calibration: compute once, reuse if cached ------------------
from scipy.special import softmax as sp_softmax

y_true_arr = np.array(y_true_ids)
probs = sp_softmax(raw_logits / T_OPTIMAL, axis=1)

# Baseline without thresholds
_base_preds = np.argmax(probs, axis=1)
_base_f1 = f1_score(y_true_arr, _base_preds, average=None, labels=list(range(N_CLASSES)))

if os.path.exists(THRESHOLD_PATH):
    thresholds = np.array([_thr_cache.get("thresholds", {}).get(cls, 1.0) for cls in classes], dtype=float)
    log("thresholds.json exists -> skipping grid search; reloading for plots and reports", "OK")
else:
    log(f"Searching optimal thresholds with grid search ({THRESHOLD_SEARCH_PASSES} passes)...")
    thresholds = np.ones(N_CLASSES)
    SEARCH_GRID = np.linspace(THRESHOLD_SEARCH_MIN, THRESHOLD_SEARCH_MAX, THRESHOLD_SEARCH_STEPS)

    for _pass in range(THRESHOLD_SEARCH_PASSES):
        _improved = 0
        for cls_idx in range(N_CLASSES):
            best_t = thresholds[cls_idx]
            best_f1 = f1_score(y_true_arr, np.argmax(probs / thresholds, axis=1), labels=[cls_idx], average=None)[0]
            for t in SEARCH_GRID:
                _t_vec = thresholds.copy()
                _t_vec[cls_idx] = t
                _preds = np.argmax(probs / _t_vec, axis=1)
                _f1 = f1_score(y_true_arr, _preds, labels=[cls_idx], average=None)[0]
                if _f1 > best_f1 + THRESHOLD_IMPROVEMENT_EPS:
                    best_f1 = _f1
                    best_t = t
            if abs(best_t - thresholds[cls_idx]) > 1e-4:
                _improved += 1
            thresholds[cls_idx] = best_t
        log(f"  Pass {_pass+1}/{THRESHOLD_SEARCH_PASSES} - {_improved} classes changed threshold")

_cal_preds = np.argmax(probs / thresholds, axis=1)
_cal_f1 = f1_score(y_true_arr, _cal_preds, average=None, labels=list(range(N_CLASSES)))

cal_acc = accuracy_score(y_true_arr, _cal_preds)
cal_f1w = f1_score(y_true_arr, _cal_preds, average="weighted")
cal_f1m = f1_score(y_true_arr, _cal_preds, average="macro")

print(f"\n{'='*72}")
print(f"  {'Metric':<20}  {'Before':>10}  {'After':>10}  {'Delta':>8}")
print(f"  {'-'*68}")
for _name, _before, _after in [
    ("Accuracy", model_acc, cal_acc),
    ("F1-weighted", model_f1w, cal_f1w),
    ("F1-macro", model_f1m, cal_f1m),
]:
    _delta = _after - _before
    _sign = "+" if _delta >= 0 else ""
    print(f"  {_name:<20}  {_before:>10.4f}  {_after:>10.4f}  {_sign}{_delta:>7.4f}")
print(f"{'='*72}")

print(f"\n  {'Class':<38}  {'Before':>7}  {'After':>7}  {'Delta':>7}  {'Threshold':>10}")
print(f"  {'-'*72}")
for i, cls in enumerate(classes):
    _b = _base_f1[i]
    _a = _cal_f1[i]
    _d = _a - _b
    _sign = "+" if _d >= 0 else ""
    _mark = " ^" if _d > 0.005 else (" v" if _d < -0.005 else "")
    print(f"  {cls:<38}  {_b:>7.4f}  {_a:>7.4f}  {_sign}{_d:>6.4f}  {thresholds[i]:>10.4f}{_mark}")


In [ ]:
# -- 8.2 Per-class F1 delta and threshold ----------------------------------------
_items = []
for i, cls in enumerate(classes):
    _items.append({
        "class": cls,
        "f1_before": float(_base_f1[i]),
        "f1_after": float(_cal_f1[i]),
        "delta": float(_cal_f1[i] - _base_f1[i]),
        "threshold": float(thresholds[i]),
    })
_df = pd.DataFrame(_items)

_df_delta = _df.sort_values("delta", ascending=False).reset_index(drop=True)
_df_thr = _df.sort_values("threshold", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 9), gridspec_kw={"width_ratios": [1.2, 1.1]})

# Delta F1
_colors_d = ["#2a9d8f" if v >= 0 else "#e76f51" for v in _df_delta["delta"]]
_bd = axes[0].barh(_df_delta["class"], _df_delta["delta"], color=_colors_d, edgecolor="white")
axes[0].invert_yaxis()
for b, v in zip(_bd, _df_delta["delta"]):
    axes[0].text(v + (0.0008 if v >= 0 else -0.0008), b.get_y() + b.get_height()/2,
                 f"{v:+.4f}", va="center", ha="left" if v >= 0 else "right", fontsize=8)
axes[0].axvline(0, color="#555", lw=1)
axes[0].set_title("Calibration helps most on some weak classes", fontweight="bold")
axes[0].set_xlabel("Delta F1 (after - before)")
axes[0].grid(axis="x", alpha=0.25)

# Threshold by class
_colors_t = ["#e76f51" if t > 1.0 else "#2a9d8f" if t < 1.0 else "#6c8ebf" for t in _df_thr["threshold"]]
_bt = axes[1].barh(_df_thr["class"], _df_thr["threshold"], color=_colors_t, edgecolor="white")
axes[1].invert_yaxis()
for b, v in zip(_bt, _df_thr["threshold"]):
    axes[1].text(v + 0.02, b.get_y() + b.get_height()/2, f"{v:.3f}", va="center", fontsize=8)
axes[1].axvline(1.0, color="#1f77b4", ls="--", lw=1.8, label="Threshold = 1.0")
axes[1].set_title("Threshold > 1 tightens a class; < 1 loosens it", fontweight="bold")
axes[1].set_xlabel("Threshold")
axes[1].grid(axis="x", alpha=0.25)
axes[1].legend(loc="lower right", fontsize=9)

fig.tight_layout()
save_fig(fig, "06_threshold_calibration.png")

In [ ]:
# -- 8.3 Save thresholds only when cache missing; otherwise just sync config ----
if not os.path.exists(THRESHOLD_PATH):
    _threshold_config = {
        "temperature": T_OPTIMAL,
        "method": "per_class_f1_grid_search",
        "search_range": [THRESHOLD_SEARCH_MIN, THRESHOLD_SEARCH_MAX],
        "n_grid": THRESHOLD_SEARCH_STEPS,
        "n_passes": THRESHOLD_SEARCH_PASSES,
        "usage": "predict = argmax(softmax(logits) / thresholds)",
        "thresholds": {cls: float(thresholds[i]) for i, cls in enumerate(classes)},
        "metrics_before": {"accuracy": model_acc, "f1_weighted": model_f1w, "f1_macro": model_f1m},
        "metrics_after": {"accuracy": cal_acc, "f1_weighted": cal_f1w, "f1_macro": cal_f1m},
    }
    with open(THRESHOLD_PATH, "w", encoding="utf-8") as f:
        json.dump(_threshold_config, f, ensure_ascii=False, indent=2)
    log(f"Saved thresholds: {THRESHOLD_PATH}", "SAVE")
else:
    log(f"Threshold cache exists: {THRESHOLD_PATH} -> not overwriting", "OK")

# Update label_config.json so the app knows thresholds exist
with open(LABEL_CONFIG_PATH, "r", encoding="utf-8") as f:
    _lc = json.load(f)
_lc["threshold_file"] = THRESHOLD_FILE_NAME
_lc["use_threshold_calibration"] = True
with open(LABEL_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(_lc, f, ensure_ascii=False, indent=2)
log("Updated label_config.json with threshold_file", "SAVE")

# Sync the final Hugging Face artifact for model-service upload/activation.
if ACTIVE_ARTIFACT_DIR.exists():
    shutil.rmtree(ACTIVE_ARTIFACT_DIR)
shutil.copytree(
    MODEL_DIR,
    ACTIVE_ARTIFACT_DIR,
    ignore=shutil.ignore_patterns("train_history.pkl"),
)
log(f"Synced active artifact: {ACTIVE_ARTIFACT_DIR}", "SAVE")

print(f"\n  [THRESHOLDS PER CLASS]")
print(f"  {'Class':<38}  {'Threshold':>10}  {'Meaning'}")
print(f"  {'-'*70}")
for i, cls in enumerate(classes):
    t = thresholds[i]
    if t > 1.2:
        meaning = "Reduce over-prediction"
    elif t < 0.8:
        meaning = "Reduce under-prediction"
    else:
        meaning = "Almost unchanged"
    print(f"  {cls:<38}  {t:>10.4f}  {meaning}")


---
## Section 9 - Diagnostics
This section prints model-improvement signals after training and calibration. It does not fine-tune the model.

In [ ]:
# -- 9.1 Build before/after diagnostic table ----------------------------------
if "_cal_preds" not in dir():
    probs = sp_softmax(raw_logits / T_OPTIMAL, axis=1)
    _cal_preds = np.argmax(probs / thresholds, axis=1)

_base_report = classification_report(
    y_true_ids, y_pred_ids, target_names=classes, digits=4,
    output_dict=True, zero_division=0
)
_cal_report = classification_report(
    y_true_ids, _cal_preds, target_names=classes, digits=4,
    output_dict=True, zero_division=0
)
_diag_df = pd.DataFrame([
    {
        "class": cls,
        "support": int(_base_report[cls]["support"]),
        "precision_before": _base_report[cls]["precision"],
        "recall_before":    _base_report[cls]["recall"],
        "f1_before":        _base_report[cls]["f1-score"],
        "precision_after":  _cal_report[cls]["precision"],
        "recall_after":     _cal_report[cls]["recall"],
        "f1_after":         _cal_report[cls]["f1-score"],
        "threshold":        float(thresholds[classes.index(cls)]),
    }
    for cls in classes
])
_diag_df["delta_f1"] = _diag_df["f1_after"] - _diag_df["f1_before"]
_diag_df["pr_gap_after"] = _diag_df["recall_after"] - _diag_df["precision_after"]
_diag_df["support_pct"] = _diag_df["support"] / max(int(_diag_df["support"].sum()), 1)

print(f"\n{'='*92}")
print("[PHOBERT DIAGNOSTIC SUMMARY]")
print(f"  F1-macro before : {model_f1m:.4f}")
print(f"  F1-macro after  : {cal_f1m:.4f}  ({cal_f1m - model_f1m:+.4f})")
print(f"  F1-weighted bef.: {model_f1w:.4f}")
print(f"  F1-weighted aft.: {cal_f1w:.4f}  ({cal_f1w - model_f1w:+.4f})")
print(f"  Accuracy before : {model_acc:.4f}")
print(f"  Accuracy after  : {cal_acc:.4f}  ({cal_acc - model_acc:+.4f})")
print(f"  Temperature     : {T_OPTIMAL:.4f}")
print(f"{'='*92}")

print("\n[5 WEAKEST CLASSES AFTER CALIBRATION]")
for _, r in _diag_df.sort_values(["f1_after", "support"], ascending=[True, True]).head(5).iterrows():
    print(f"  {r['class']:<30}  F1={r['f1_after']:.4f}  P={r['precision_after']:.4f}  R={r['recall_after']:.4f}  support={int(r['support'])}")

print("\n[5 LOWEST-SUPPORT CLASSES]")
for _, r in _diag_df.sort_values(["support", "f1_after"], ascending=[True, True]).head(5).iterrows():
    print(f"  {r['class']:<30}  support={int(r['support']):>5}  ({r['support_pct']*100:>5.2f}%)  F1_after={r['f1_after']:.4f}")

print("\n[5 MOST IMPROVED CLASSES AFTER CALIBRATION]")
for _, r in _diag_df.sort_values("delta_f1", ascending=False).head(5).iterrows():
    print(f"  {r['class']:<30}  before={r['f1_before']:.4f}  after={r['f1_after']:.4f}  delta={r['delta_f1']:+.4f}  thr={r['threshold']:.4f}")

print("\n[5 MOST REGRESSED CLASSES AFTER CALIBRATION]")
for _, r in _diag_df.sort_values("delta_f1", ascending=True).head(5).iterrows():
    print(f"  {r['class']:<30}  before={r['f1_before']:.4f}  after={r['f1_after']:.4f}  delta={r['delta_f1']:+.4f}  thr={r['threshold']:.4f}")

print("\n[LOW-RECALL CLASSES AFTER CALIBRATION]")
for _, r in _diag_df.sort_values("pr_gap_after", ascending=True).head(5).iterrows():
    print(f"  {r['class']:<30}  gap={r['pr_gap_after']:+.4f}  P={r['precision_after']:.4f}  R={r['recall_after']:.4f}  thr={r['threshold']:.4f}")

print("\n[LOW-PRECISION CLASSES AFTER CALIBRATION]")
for _, r in _diag_df.sort_values("pr_gap_after", ascending=False).head(5).iterrows():
    print(f"  {r['class']:<30}  gap={r['pr_gap_after']:+.4f}  P={r['precision_after']:.4f}  R={r['recall_after']:.4f}  thr={r['threshold']:.4f}")

# -- 9.2 Top confusion pairs after calibration -------------------------------
_cm_cal = confusion_matrix(y_true_ids, _cal_preds, labels=list(range(N_CLASSES)))
_pairs = []
for i, true_cls in enumerate(classes):
    _support = max(int(_cm_cal[i].sum()), 1)
    for j, pred_cls in enumerate(classes):
        if i == j or _cm_cal[i, j] == 0:
            continue
        _pairs.append({
            "count": int(_cm_cal[i, j]),
            "rate":  float(_cm_cal[i, j] / _support),
            "true":  true_cls,
            "pred":  pred_cls,
        })
_pairs_df = pd.DataFrame(_pairs).sort_values(["count", "rate"], ascending=[False, False]) if _pairs else pd.DataFrame(columns=["count", "rate", "true", "pred"])

print("\n[TOP 8 CONFUSION PAIRS AFTER CALIBRATION]")
if len(_pairs_df) == 0:
    print("  No off-diagonal confusion pairs.")
else:
    for _, r in _pairs_df.head(8).iterrows():
        print(f"  {r['true']:<28} -> {r['pred']:<28}  {int(r['count']):>4} times  ({r['rate']*100:>5.2f}%)")

# -- 9.3 Next-step hints -----------------------------------------------------
print("\n[NEXT-STEP HINTS FOR PHOBERT]")
if (cal_f1w - cal_f1m) > 0.06:
    print("  - Weighted F1 is still much higher than macro F1, so class imbalance remains; prioritize low-support and specialized classes.")
else:
    print("  - Weighted and macro F1 are close; focus on the strongest confusion pairs.")

_thr_high = _diag_df[_diag_df["threshold"] > 1.4].sort_values("threshold", ascending=False)["class"].tolist()
_thr_low = _diag_df[_diag_df["threshold"] < 0.8].sort_values("threshold")["class"].tolist()
_low_f1 = _diag_df[_diag_df["f1_after"] < 0.80].sort_values("f1_after")["class"].tolist()

if _thr_high:
    print(f"  - Classes with high thresholds: {', '.join(_thr_high[:6])}{' ...' if len(_thr_high) > 6 else ''}")
    print("    Check whether these classes are truly over-predicted; if recall drops too much, lower the threshold or add data.")
if _thr_low:
    print(f"  - Classes with low thresholds: {', '.join(_thr_low[:6])}{' ...' if len(_thr_low) > 6 else ''}")
    print("    If precision is still low here, calibration is only a surface fix; review data, class weights, and confusion pairs.")
if _low_f1:
    print(f"  - Classes with F1_after < 0.80: {', '.join(_low_f1[:6])}{' ...' if len(_low_f1) > 6 else ''}")
    print("    Prioritize adding real samples, reviewing labels, or carefully increasing class weights for these classes.")

print("  - If a class improves mainly from threshold tuning, the base model features may still be weak; do not rely on calibration alone long term.")
print("  - If the same confusion pairs repeat often, inspect those texts directly; data fixes usually beat broad hyperparameter tuning.")
print("  - If macro F1 remains low after calibration, keep the current LR/epoch and try careful class boosts, sampling, or more data for rare classes.")